#### Generate the sample staging customer data to test the SCD ####
**<mark>Generating this synthetic customer data as template for our production template for SCD Type handling of our dimensions</mark>**

Because we have two separate areas, here we generate the data to the staging, 
and in Slowly Changing Dim Notebooks, we can run the **MERGE** 

We can change values to test here, and then go to the other notebook to test


In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 3, Finished, Available, Finished, False)

## <span style="background-color:pink;"> Modify Tracked Columns for SCD Type 2 Test ##

##### **<mark>SCD Type 2 Data manipulation test - varying values</mark>**

In [2]:
import random

num_changes = 20
rows_to_change = random.sample(range(200), num_changes)


StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 4, Finished, Available, Finished, False)

#### Modify the columns tracked by SCD ####

**E.g. SCD columns are:</mark>**

In [3]:
scd_columns = ["customer_name", "customer_status", "address"]


StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 5, Finished, Available, Finished, False)

<mark>Read delta file from Lakehouse, to Spark DataFrame, 

<mark>Then we need to convert it to **Pandas** DataFrame, and the we can use various Pandas functions to manipulate values for our test</mark>

In [4]:
source_path = "Files/staging/customers"

df_source = spark.read.format("delta").load(source_path)


StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 6, Finished, Available, Finished, False)

In [5]:
df_source.orderBy("customer_id").show(40)

StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 7, Finished, Available, Finished, False)

+-----------+-------------------+---------------+----------+--------------------+----------+
|customer_id|      customer_name|customer_status|   address|     effective_start|is_current|
+-----------+-------------------+---------------+----------+--------------------+----------+
|      C0001|      Customer_0001|         Active| 1 Main St|2026-02-04 23:14:...|      true|
|      C0002|      Customer_0002|       Inactive| 2 Main St|2026-02-04 23:14:...|      true|
|      C0003|      Customer_0003|         Active| 3 Main St|2026-02-04 23:14:...|      true|
|      C0004|      Customer_0004|         Active| 4 Main St|2026-02-04 23:14:...|      true|
|      C0005|Customer_0005_v2_v2|       Inactive|520 New St|2026-02-04 23:14:...|      true|
|      C0006|Customer_0006_v2_v2|       Inactive|976 New St|2026-02-04 23:14:...|      true|
|      C0007|      Customer_0007|         Active| 7 Main St|2026-02-04 23:14:...|      true|
|      C0008|      Customer_0008|       Inactive| 8 Main St|2026-02-04

##### Convert **Spark** DataFrame to <mark>Pandas **DataFrame**</mark> #####

In [6]:
pdf = df_source.toPandas()


StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 8, Finished, Available, Finished, False)

##### <mark>Make random changes now to the columns we will be tracking for our SCD Type 2 in Pandas DataFrame, like</mark>: #####

In [7]:
for idx in rows_to_change:
    # Change customer_status randomly
    pdf.loc[idx, "customer_status"] = np.random.choice(["Active", "Inactive"])
    
    # Change address slightly
    pdf.loc[idx, "address"] = f"{random.randint(100,999)} New St"
    
    # Optionally change name for testing
    pdf.loc[idx, "customer_name"] = f"{pdf.loc[idx,'customer_name']}_v2"


StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 9, Finished, Available, Finished, False)

##### Reset the dates and tracking flags once we've done our changes for the SCD scenario we are testing for in the dimension #####

In [8]:
from datetime import datetime

pdf["effective_start"] = datetime.now()
pdf["effective_end"] = None
pdf["is_current"] = True

StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 10, Finished, Available, Finished, False)

#### Convert back to **Spark DataFrame** ####

In [9]:
df_source_modified = spark.createDataFrame(pdf)

StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 11, Finished, Available, Finished, False)

### Save to your staging folder ###

In [10]:
source_path = "Files/staging/customers"
df_source_modified.write.format("delta").mode("overwrite").save(source_path)


StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 12, Finished, Available, Finished, False)

##### <span style="background-color:pink;">Let's view what changes we've done for our SCD test </mark>

In [11]:
df_source_modified.orderBy("customer_id").show(40)

StatementMeta(, 9d98d0b2-e722-4853-81cd-23f0968cf40f, 13, Finished, Available, Finished, False)

+-----------+--------------------+---------------+----------+--------------------+----------+-------------+
|customer_id|       customer_name|customer_status|   address|     effective_start|is_current|effective_end|
+-----------+--------------------+---------------+----------+--------------------+----------+-------------+
|      C0001|       Customer_0001|         Active| 1 Main St|2026-02-05 00:11:...|      true|         NULL|
|      C0002|       Customer_0002|       Inactive| 2 Main St|2026-02-05 00:11:...|      true|         NULL|
|      C0003|       Customer_0003|         Active| 3 Main St|2026-02-05 00:11:...|      true|         NULL|
|      C0004|       Customer_0004|         Active| 4 Main St|2026-02-05 00:11:...|      true|         NULL|
|      C0005| Customer_0005_v2_v2|       Inactive|520 New St|2026-02-05 00:11:...|      true|         NULL|
|      C0006| Customer_0006_v2_v2|       Inactive|976 New St|2026-02-05 00:11:...|      true|         NULL|
|      C0007|       Customer

#### Then run the  **Slowly Changing Dim Notebook** to test SCD Type 2 ####